# Natural-prior and minority-coverage preparation
Run cells in order. Durable artifacts are written to the configured Drive results folder.


In [22]:
from pathlib import Path
import os, sys, json, errno, time
DRIVE = Path("/content/drive/MyDrive")
try:
    from google.colab import drive
except ImportError:
    drive = None

def with_drive_retry(operation, attempts=3):
    for attempt in range(attempts):
        try:
            return operation()
        except OSError as exc:
            disconnected = exc.errno in {errno.ENOTCONN, errno.EIO, errno.ESTALE} or "Transport endpoint" in str(exc)
            if not disconnected or attempt == attempts - 1:
                raise
            print(f"Drive disconnected; remounting before retry {attempt + 1}/{attempts - 1}.", flush=True)
            if drive is not None:
                drive.mount("/content/drive", force_remount=True)
            time.sleep(1)

def ensure_drive():
    if drive is not None:
        if not DRIVE.is_dir():
            drive.mount("/content/drive", force_remount=False)
        next(DRIVE.iterdir(), None)
with_drive_retry(ensure_drive)

def resolve_v2():
    configured = os.environ.get("ATLAS_V2_ROOT")
    candidates = ([Path(configured).expanduser()] if configured else []) + [
        DRIVE/"projects/implementation/ATLAS-IDNet/implementation/v2",
        Path.cwd(), *Path.cwd().parents,
    ]
    for candidate in candidates:
        if all((candidate/name).is_file() for name in (
            "src/atlas_v2/data.py", "src/atlas_v2/models.py",
            "src/atlas_v2/experiments.py", "src/atlas_v2/agent.py", "requirements.txt"
        )):
            return candidate.resolve()
    raise FileNotFoundError("Synchronized implementation/v2 source is not visible to this kernel.")

V2 = with_drive_retry(resolve_v2)
sys.path.insert(0, str(V2/"src"))
for module in list(sys.modules):
    if module == "atlas_v2" or module.startswith("atlas_v2."):
        del sys.modules[module]
import atlas_v2
print("Code:", V2)


Code: /content/drive/MyDrive/projects/implementation/ATLAS-IDNet/implementation/v2


In [23]:
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(V2/"requirements.txt")])
import numpy as np, pandas as pd, sklearn, scipy, torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CPU count:", os.cpu_count())


PyTorch: 2.11.0+cpu
CUDA available: False
CPU count: 2


In [24]:
BASE = DRIVE/"projects/Dataset/Security" if DRIVE.exists() else Path("/Volumes/LocalDrive1/Dataset/Security")
RESULTS = DRIVE/"projects/implementation/ATLAS-IDNet/results/v2" if DRIVE.exists() else V2/"results"
RESULTS.mkdir(parents=True, exist_ok=True)
common = {
    "split": "stratified", "group_column": None, "time_column": None,
    "exclude_columns": ["Label", "label", "Attack", "Attack_type", "Class", "Dataset", "dataset",
                        "IPV4_SRC_ADDR", "IPV4_DST_ADDR", "Flow ID", "Src IP", "Dst IP", "Timestamp"]
}
datasets = {
    "CICIoT2023": {"csv_globs": [str(BASE/"CICIoT2023/**/*.csv")], "label": "label"},
    "NF-UQ-NIDS-v2": {"csv_globs": [str(BASE/"NF-UQ-NIDS-v2/**/*.csv")], "label": "Attack", "conflicting_duplicate_policy": "drop_all"},
}
protocols = {}
for dataset_name, dataset in datasets.items():
    natural = {**dataset, **common, "max_rows": 500000}
    minority = {**dataset, **common, "max_rows": 500000, "class_cap": 5000}
    protocols[dataset_name] = {"natural": natural, "minority_challenge": minority}
PUBLICATION_CONFIG = {
    "results": str(RESULTS),
    "seeds": [42, 43, 44, 45, 46],
    "challenge_seeds": [42, 43, 44, 45, 46],
    "epochs": 30,
    "width": 128,
    "batch_size": 512,
    "patience": 7,
    "loss": "balanced",
    "challenge_support_policy": {
        "required_minimum_per_heldout_partition": 15,
        "preferred_minimum_per_heldout_partition": 30,
        "below_preferred_interpretation": "source-limited; retain the class, report uncertainty, and avoid strong class-specific claims",
    },
    "protocols": protocols,
    "submitted_reference": {
        "CICIoT2023": {"accuracy": 0.9503, "weighted_f1": 0.9462, "macro_f1": 0.6460},
        "NF-UQ-NIDS-v2": {"accuracy": 0.9795, "weighted_f1": 0.9787, "macro_f1": 0.7154},
        "provenance": "Previously submitted manuscript; target only, not an independently reproduced baseline."
    }
}
PUBLICATION_CONFIG_PATH = RESULTS/"publication_config.json"
temporary = PUBLICATION_CONFIG_PATH.with_suffix(".tmp")
temporary.write_text(json.dumps(PUBLICATION_CONFIG, indent=2))
os.replace(temporary, PUBLICATION_CONFIG_PATH)
print("Publication config:", PUBLICATION_CONFIG_PATH)
print(json.dumps(PUBLICATION_CONFIG, indent=2))


Publication config: /content/drive/MyDrive/projects/implementation/ATLAS-IDNet/results/v2/publication_config.json
{
  "results": "/content/drive/MyDrive/projects/implementation/ATLAS-IDNet/results/v2",
  "seeds": [
    42,
    43,
    44,
    45,
    46
  ],
  "challenge_seeds": [
    42,
    43,
    44,
    45,
    46
  ],
  "epochs": 30,
  "width": 128,
  "batch_size": 512,
  "patience": 7,
  "loss": "balanced",
  "challenge_support_policy": {
    "required_minimum_per_heldout_partition": 15,
    "preferred_minimum_per_heldout_partition": 30,
    "below_preferred_interpretation": "source-limited; retain the class, report uncertainty, and avoid strong class-specific claims"
  },
  "protocols": {
    "CICIoT2023": {
      "natural": {
        "csv_globs": [
          "/content/drive/MyDrive/projects/Dataset/Security/CICIoT2023/**/*.csv"
        ],
        "label": "label",
        "split": "stratified",
        "group_column": null,
        "time_column": null,
        "exclude_columns

In [25]:
from atlas_v2.data import PREPARATION_VERSION, prepare
import hashlib
support_policy = PUBLICATION_CONFIG["challenge_support_policy"]
PUBLICATION_CONFIG["preparation_support_audit"] = {}
for dataset_name, dataset_protocols in PUBLICATION_CONFIG["protocols"].items():
    for protocol_name, spec in dataset_protocols.items():
        clean_spec = {key: value for key, value in spec.items() if key != "prepared"}
        key = hashlib.sha256(json.dumps({"spec": clean_spec, "version": PREPARATION_VERSION}, sort_keys=True).encode()).hexdigest()[:12]
        folder = RESULTS/dataset_name/f"prepared_{protocol_name}_{key}"
        if not (folder/"metadata.json").exists():
            with_drive_retry(lambda: prepare(clean_spec, folder, seed=42))
        metadata = json.loads((folder/"metadata.json").read_text())
        PUBLICATION_CONFIG["protocols"][dataset_name][protocol_name]["prepared"] = str(folder)
        print(dataset_name, protocol_name, metadata["counts"])
        if protocol_name == "minority_challenge" and not metadata["full_class_evaluation_supported"]:
            raise RuntimeError(f"{dataset_name}: minority challenge lacks held-out support: {metadata['missing_classes']}")
        if protocol_name == "minority_challenge":
            minimum = metadata["minimum_heldout_support"]
            required = support_policy["required_minimum_per_heldout_partition"]
            preferred = support_policy["preferred_minimum_per_heldout_partition"]
            if minimum < required:
                raise RuntimeError(
                    f"{dataset_name}: minority challenge has only {minimum} records in its least-supported "
                    f"held-out class, below the explicit minimum of {required}."
                )
            status = "adequate" if minimum >= preferred else "source_limited"
            PUBLICATION_CONFIG["preparation_support_audit"][f"{dataset_name}/{protocol_name}"] = {
                "minimum_heldout_support": minimum,
                "required_minimum": required,
                "preferred_minimum": preferred,
                "status": status,
                "interpretation": ("standard challenge support" if status == "adequate" else
                    support_policy["below_preferred_interpretation"]),
            }
            if status == "source_limited":
                print(
                    f"WARNING: {dataset_name} minority challenge is source-limited at {minimum} held-out records; "
                    "the class is retained, but its class-specific estimate requires uncertainty reporting.",
                    flush=True,
                )
        temporary = PUBLICATION_CONFIG_PATH.with_suffix(".tmp")
        temporary.write_text(json.dumps(PUBLICATION_CONFIG, indent=2))
        os.replace(temporary, PUBLICATION_CONFIG_PATH)


CICIoT2023 natural {'train': [24, 8331, 40, 37, 2108, 218, 53687, 3362, 30578, 30386, 30403, 170, 27100, 33780, 40698, 2147, 1303, 101, 551, 15265, 20130, 25008, 2301, 7366, 5674, 6548, 995, 726, 12, 596, 40, 10, 275, 31], 'val': [4, 1190, 6, 5, 301, 31, 7670, 480, 4368, 4341, 4343, 24, 3872, 4826, 5814, 307, 186, 14, 79, 2181, 2876, 3572, 329, 1052, 811, 936, 142, 104, 2, 85, 6, 2, 39, 4], 'test': [7, 2380, 11, 11, 602, 62, 15339, 960, 8736, 8682, 8686, 49, 7743, 9652, 11628, 614, 372, 29, 157, 4361, 5751, 7145, 657, 2104, 1621, 1871, 284, 208, 4, 170, 11, 3, 78, 9]}
CICIoT2023 minority_challenge {'train': [2252, 3500, 3500, 3500, 3500, 3500, 3500, 3500, 3500, 3500, 3500, 3500, 3500, 3500, 3500, 3500, 3500, 3500, 3500, 3500, 3500, 3500, 3500, 3500, 3500, 3500, 3500, 3500, 1584, 3500, 3500, 877, 3500, 2692], 'val': [322, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 226, 500, 500, 125, 500, 385], '

In [26]:
audit_rows = []
support_policy = PUBLICATION_CONFIG["challenge_support_policy"]
for dataset_name, dataset_protocols in PUBLICATION_CONFIG["protocols"].items():
    for protocol_name, spec in dataset_protocols.items():
        metadata = json.loads((Path(spec["prepared"])/"metadata.json").read_text())
        audit_rows.append({
            "dataset": dataset_name, "protocol": protocol_name,
            "rows_scanned": metadata["source_rows_scanned"],
            "retained_rows": sum(sum(values) for values in metadata["counts"].values()),
            "features": len(metadata["features"]), "classes": len(metadata["classes"]),
            "duplicates_removed": metadata["duplicates_removed"],
            "ambiguous_duplicate_groups": metadata.get("conflicting_duplicates", 0),
            "ambiguous_rows_removed": metadata.get("ambiguous_rows_removed", 0),
            "minimum_heldout_support": metadata["minimum_heldout_support"],
            "required_minimum_heldout_support": (support_policy["required_minimum_per_heldout_partition"]
                                                  if protocol_name == "minority_challenge" else None),
            "preferred_minimum_heldout_support": (support_policy["preferred_minimum_per_heldout_partition"]
                                                   if protocol_name == "minority_challenge" else None),
            "support_status": (PUBLICATION_CONFIG.get("preparation_support_audit", {})
                               .get(f"{dataset_name}/{protocol_name}", {}).get("status", "natural_prior")),
            "full_class_evaluation_supported": metadata["full_class_evaluation_supported"],
            "sampling_protocol": metadata["sampling_protocol"],
        })
audit = pd.DataFrame(audit_rows)
audit.to_csv(RESULTS/"preparation_audit.csv", index=False)
display(audit)


,dataset,protocol,rows_scanned,retained_rows,features,classes,duplicates_removed,ambiguous_duplicate_groups,ambiguous_rows_removed,minimum_heldout_support,required_minimum_heldout_support,preferred_minimum_heldout_support,support_status,full_class_evaluation_supported,sampling_protocol
0,CICIoT2023,natural,46686579,500000,46,34,0,0,0,2,NaN,NaN,natural_prior,True,uniform sample
1,CICIoT2023,minority_challenge,46686579,160578,46,34,0,0,0,125,15.0,30.0,adequate,True,class-capped; altered class priors; separate c...
2,NF-UQ-NIDS-v2,natural,75987976,496904,41,20,3076,9,20,1,NaN,NaN,natural_prior,True,uniform sample
3,NF-UQ-NIDS-v2,minority_challenge,75987976,83427,41,21,2580,390,3739,16,15.0,30.0,source_limited,True,class-capped; altered class priors; separate c...
